In [ ]:
import os
import re
import csv
import json
import time
from datetime import datetime
from pathlib import Path
from typing import Optional
from groq import Groq
from pydantic import BaseModel, ConfigDict, ValidationError, conint

try:
    import mlflow
except ImportError:
    mlflow = None

In [3]:
Score = conint(ge=0, le=5)


class EvidenceItem(BaseModel):
    model_config = ConfigDict(extra="forbid")
    subscore: str
    quote: str
    reason: str


class MotivationSubscores(BaseModel):
    model_config = ConfigDict(extra="forbid")
    university_specificity: Score
    program_fit: Score
    goal_alignment: Score
    intrinsic_motivation: Score
    specificity_of_reasoning: Score


class LeadershipPotentialSubscores(BaseModel):
    model_config = ConfigDict(extra="forbid")
    leadership_definition_quality: Score
    concrete_example_presence: Score
    initiative: Score
    responsibility: Score
    impact: Score
    reflection: Score


class ResponseStructureSubscores(BaseModel):
    model_config = ConfigDict(extra="forbid")
    clarity: Score
    coherence: Score
    completeness: Score
    relevance: Score
    conciseness: Score


class Motivation(BaseModel):
    model_config = ConfigDict(extra="forbid")
    subscores: MotivationSubscores
    evidence: list[EvidenceItem]
    weaknesses: list[str]


class LeadershipPotential(BaseModel):
    model_config = ConfigDict(extra="forbid")
    subscores: LeadershipPotentialSubscores
    evidence: list[EvidenceItem]
    weaknesses: list[str]


class ResponseStructure(BaseModel):
    model_config = ConfigDict(extra="forbid")
    subscores: ResponseStructureSubscores
    evidence: list[EvidenceItem]
    weaknesses: list[str]


class ContextNotes(BaseModel):
    model_config = ConfigDict(extra="forbid")
    family_support_context: str
    encouragement_source: str


class TranscriptScoringResult(BaseModel):
    model_config = ConfigDict(extra="forbid")
    motivation: Motivation
    leadership_potential: LeadershipPotential
    response_structure: ResponseStructure
    context_notes: ContextNotes
    risk_flags: list[str]
    missing_evidence: list[str]


def extract_json_object(text: str) -> dict:
    content = text.strip()
    if content.startswith("```"):
        content = re.sub(r"^```(?:json)?\s*|\s*```$", "", content, flags=re.IGNORECASE | re.DOTALL).strip()

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", content, flags=re.DOTALL)
        if not match:
            raise ValueError("Ответ модели не содержит корректный JSON-объект")
        return json.loads(match.group(0))


def get_client() -> Groq:
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError("GROQ_API_KEY не задан в окружении")
    return Groq(api_key=api_key)


def generate_response(
    prompt: str,
    text: str,
    model: Optional[str] = None,
    max_retries: int = 2,
    temperature: float = 0.2,
) -> dict:
    if not prompt.strip():
        raise ValueError("prompt не должен быть пустым")
    if not text.strip():
        raise ValueError("text не должен быть пустым")
    if max_retries < 0:
        raise ValueError("max_retries не может быть отрицательным")

    client = get_client()
    model_name = model or os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

    last_error: Optional[Exception] = None
    for attempt in range(max_retries + 1):
        try:
            completion = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": prompt},
                    {"role": "user", "content": text},
                ],
                temperature=temperature,
            )

            answer = completion.choices[0].message.content if completion.choices else ""
            payload = extract_json_object(answer or "")
            validated = TranscriptScoringResult.model_validate(payload)

            return {
                "answer": validated.model_dump(),
                "model": model_name,
                "attempt": attempt + 1,
            }
        except (json.JSONDecodeError, ValueError, ValidationError) as error:
            last_error = error
            if attempt == max_retries:
                break
            continue
        except Exception as error:
            raise RuntimeError(f"Ошибка Groq API: {error}") from error

    raise RuntimeError(
        f"Не удалось получить валидный JSON после {max_retries + 1} попыток: {last_error}"
    )

In [4]:
def read_text_file(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Файл не найден: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def load_env_file(path: str = ".env") -> None:
    if not os.path.exists(path):
        return
    with open(path, "r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip('"').strip("'")
            if key and key not in os.environ:
                os.environ[key] = value


def get_temperature(default: float = 0.2) -> float:
    raw_value = os.getenv("GROQ_TEMPERATURE")
    if raw_value is None or not raw_value.strip():
        return default
    try:
        return float(raw_value)
    except ValueError as error:
        raise ValueError("TEMPERATURE в .env должен быть числом, например 0.2") from error


load_env_file()
prompt_text = read_text_file("prompt.txt")
input_text = read_text_file("input.txt")
temperature = get_temperature()

print(f"Файлы загружены. TEMPERATURE={temperature}, MODEL = {os.getenv('GROQ_MODEL')}")

Файлы загружены. TEMPERATURE=0.2, MODEL = openai/gpt-oss-20b


In [80]:
result = generate_response(
    prompt=prompt_text,
    text=input_text,
    model=None,
    max_retries=3,
    temperature=temperature,
 )
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "answer": {
    "motivation": {
      "subscores": {
        "university_specificity": 4,
        "program_fit": 4,
        "goal_alignment": 4,
        "intrinsic_motivation": 3,
        "specificity_of_reasoning": 4
      },
      "evidence": [
        {
          "subscore": "university_specificity",
          "quote": "the reason I am applying to inVision U is because I want to study in an environment where I can not only learn technical skills but also actually build real products. I read about the program and I saw that it focuses on product thinking, entrepreneurship, and real-world impact, and that really matches what I want to do in the future.",
          "reason": "Directly ties personal desire to specific attributes of inVision U."
        },
        {
          "subscore": "program_fit",
          "quote": "I am particularly interested in the Innovative IT Product Design and Development program. ... I tried to build a small app for my school to help students organize t

In [81]:
result_count = {}
sections = ("motivation", "leadership_potential", "response_structure")
for section in sections:
    subscores = result["answer"][section]["subscores"]
    result_count[section] = sum(subscores.values())

result_count["total"] = sum(result_count.values())
print(json.dumps(result_count, ensure_ascii=False, indent=2))

if mlflow is None:
    print("MLflow не установлен. Для логирования выполните: pip install mlflow")
else:
    mlflow.set_experiment("transcript_llm_comparison")
    run_name = f"single_input__{result['model']}__{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "mode": "single_input",
            "model_name": result["model"],
            "attempt": result["attempt"],
            "schema_version": "transcript_scoring_v1",
        })
        for key, value in result_count.items():
            mlflow.log_metric(f"result_count_{key}", value)
        mlflow.log_dict(result, "single_prediction.json")
        print("MLflow: result_count и ответ залогированы в run")

{
  "motivation": 19,
  "leadership_potential": 26,
  "response_structure": 22,
  "total": 67
}


2026/03/28 16:33:49 INFO mlflow.tracking.fluent: Experiment with name 'transcript_llm_comparison' does not exist. Creating a new experiment.


MLflow: result_count и ответ залогированы в run


In [ ]:
TESTS = [
    {"id": "good_1", "label": "good", "path": "test_inputs/good_candidate_1.txt"},
    {"id": "good_2", "label": "good", "path": "test_inputs/good_candidate_2.txt"},
    {"id": "mid_1", "label": "middle", "path": "test_inputs/middle_candidate_1.txt"},
    {"id": "mid_2", "label": "middle", "path": "test_inputs/middle_candidate_2.txt"},
    {"id": "bad_1", "label": "bad", "path": "test_inputs/bad_candidate_1.txt"},
    {"id": "bad_2", "label": "bad", "path": "test_inputs/bad_candidate_2.txt"},
]


def compute_result_count(answer: dict) -> dict:
    counts = {}
    sections = ("motivation", "leadership_potential", "response_structure")
    for section in sections:
        counts[section] = sum(answer[section]["subscores"].values())
    counts["total"] = sum(counts.values())
    return counts


def run_single_test_case(test_item: dict, model_name: Optional[str], max_retries: int, temperature: float) -> dict:
    text = read_text_file(test_item["path"])
    started = time.time()
    try:
        response = generate_response(
            prompt=prompt_text,
            text=text,
            model=model_name,
            max_retries=max_retries,
            temperature=temperature,
        )
        latency_sec = time.time() - started
        counts = compute_result_count(response["answer"])
        return {
            "test_id": test_item["id"],
            "label": test_item["label"],
            "path": test_item["path"],
            "schema_pass": 1,
            "latency_sec": latency_sec,
            "model": response["model"],
            "attempt": response["attempt"],
            "result_count": counts,
            "error": None,
            "answer": response["answer"],
        }
    except Exception as error:
        latency_sec = time.time() - started
        return {
            "test_id": test_item["id"],
            "label": test_item["label"],
            "path": test_item["path"],
            "schema_pass": 0,
            "latency_sec": latency_sec,
            "model": model_name or os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile"),
            "attempt": None,
            "result_count": None,
            "error": str(error),
            "answer": None,
        }


def log_batch_to_mlflow(
    model_name: Optional[str] = None,
    prompt_version: str = "v1",
    max_retries: int = 3,
    temperature_override: Optional[float] = None,
) -> list[dict]:
    if mlflow is None:
        raise RuntimeError("MLflow не установлен. Установите: pip install mlflow")

    run_temperature = temperature if temperature_override is None else temperature_override
    resolved_model =  "meta-llama/llama-4-scout-17b-16e-instruct"   #model_name or os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
    safe_model_for_file = re.sub(r"[^A-Za-z0-9_.-]+", "_", resolved_model)

    rows = []
    for test_item in TESTS:
        rows.append(run_single_test_case(test_item, resolved_model, max_retries, run_temperature))

    passed_rows = [row for row in rows if row["schema_pass"] == 1]
    valid_json_rate = len(passed_rows) / len(rows) if rows else 0.0
    avg_latency = sum(row["latency_sec"] for row in rows) / len(rows) if rows else 0.0

    def avg_metric(metric_name: str) -> float:
        values = [row["result_count"][metric_name] for row in passed_rows if row["result_count"]]
        return (sum(values) / len(values)) if values else 0.0

    good_rows = [row for row in passed_rows if row["label"] == "good"]
    middle_rows = [row for row in passed_rows if row["label"] == "middle"]
    bad_rows = [row for row in passed_rows if row["label"] == "bad"]

    def avg_total(rows_part: list[dict]) -> float:
        totals = [row["result_count"]["total"] for row in rows_part if row["result_count"]]
        return (sum(totals) / len(totals)) if totals else 0.0

    mlflow.set_experiment("transcript_llm_comparison")
    run_name = f"{resolved_model}__{prompt_version}__{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "model_name": resolved_model,
            "prompt_version": prompt_version,
            "schema_version": "transcript_scoring_v1",
            "num_tests": len(TESTS),
            "max_retries": max_retries,
            "temperature": run_temperature,
        })

        mlflow.log_metrics({
            "valid_json_rate": valid_json_rate,
            "schema_pass_rate": valid_json_rate,
            "avg_latency_sec": avg_latency,
            "num_passed": float(len(passed_rows)),
            "num_failed": float(len(rows) - len(passed_rows)),
            "avg_motivation": avg_metric("motivation"),
            "avg_leadership": avg_metric("leadership_potential"),
            "avg_structure": avg_metric("response_structure"),
            "avg_score_total": avg_metric("total"),
            "good_avg_total": avg_total(good_rows),
            "middle_avg_total": avg_total(middle_rows),
            "bad_avg_total": avg_total(bad_rows),
        })

        outputs_dir = Path("mlflow_outputs")
        outputs_dir.mkdir(exist_ok=True)
        suffix = datetime.now().strftime("%Y%m%d_%H%M%S")
        json_path = outputs_dir / f"predictions_{safe_model_for_file}_{suffix}.json"
        csv_path = outputs_dir / f"per_test_results_{safe_model_for_file}_{suffix}.csv"
        failures_path = outputs_dir / f"failures_{safe_model_for_file}_{suffix}.json"

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(rows, f, ensure_ascii=False, indent=2)

        with open(csv_path, "w", encoding="utf-8", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                "test_id",
                "label",
                "path",
                "schema_pass",
                "latency_sec",
                "model",
                "attempt",
                "motivation",
                "leadership_potential",
                "response_structure",
                "total",
                "error",
            ])
            for row in rows:
                counts = row["result_count"] or {}
                writer.writerow([
                    row["test_id"],
                    row["label"],
                    row["path"],
                    row["schema_pass"],
                    round(row["latency_sec"], 4),
                    row["model"],
                    row["attempt"],
                    counts.get("motivation"),
                    counts.get("leadership_potential"),
                    counts.get("response_structure"),
                    counts.get("total"),
                    row["error"],
                ])

        failures = [row for row in rows if row["schema_pass"] == 0]
        with open(failures_path, "w", encoding="utf-8") as f:
            json.dump(failures, f, ensure_ascii=False, indent=2)

        mlflow.log_artifact(str(json_path))
        mlflow.log_artifact(str(csv_path))
        mlflow.log_artifact(str(failures_path))

    print(f"MLflow batch run завершён для модели: {resolved_model}")
    return rows


batch_results = log_batch_to_mlflow(model_name=None, prompt_version="v1", max_retries=3)
print(json.dumps(batch_results, ensure_ascii=False, indent=2))

MLflow batch run завершён для модели: moonshotai/kimi-k2-instruct-0905
[
  {
    "test_id": "good_1",
    "label": "good",
    "path": "test_inputs/good_candidate_1.txt",
    "schema_pass": 1,
    "latency_sec": 2.3401999473571777,
    "model": "moonshotai/kimi-k2-instruct-0905",
    "attempt": 1,
    "result_count": {
      "motivation": 21,
      "leadership_potential": 22,
      "response_structure": 25,
      "total": 68
    },
    "error": null,
    "answer": {
      "motivation": {
        "subscores": {
          "university_specificity": 4,
          "program_fit": 5,
          "goal_alignment": 4,
          "intrinsic_motivation": 4,
          "specificity_of_reasoning": 4
        },
        "evidence": [
          {
            "subscore": "university_specificity",
            "quote": "I studied your program details and I like that learning is connected to practice, teamwork, and launching real initiatives",
            "reason": "Shows specific research into inVision U's ap

In [8]:
import requests
import os

api_key = os.environ.get("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(response.json())

{'object': 'list', 'data': [{'id': 'meta-llama/llama-prompt-guard-2-86m', 'object': 'model', 'created': 1748632165, 'owned_by': 'Meta', 'active': True, 'context_window': 512, 'public_apps': None, 'max_completion_tokens': 512}, {'id': 'llama-3.3-70b-versatile', 'object': 'model', 'created': 1733447754, 'owned_by': 'Meta', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 32768}, {'id': 'meta-llama/llama-prompt-guard-2-22m', 'object': 'model', 'created': 1748632101, 'owned_by': 'Meta', 'active': True, 'context_window': 512, 'public_apps': None, 'max_completion_tokens': 512}, {'id': 'llama-3.1-8b-instant', 'object': 'model', 'created': 1693721698, 'owned_by': 'Meta', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 131072}, {'id': 'groq/compound', 'object': 'model', 'created': 1756949530, 'owned_by': 'Groq', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 8192}, {'id': 'whi